Based on Andrej Karpathy's "The spelled-out intro to language modeling: building makemore"

### Environment Setup & Data Loading

What this cell does: Loads the dataset of names into memory and inspects basic statistics. We create a list of strings (words), count the total number of names, and determine length extremes.

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

# Read all names from the dataset
words = open('names.txt', 'r').read().splitlines()

print(f"Total number of words: {len(words)}")
print(f"Shortest word length : {min(len(w) for w in words)}")
print(f"Longest word length  : {max(len(w) for w in words)}")
print(f"First 5 words        : {words[:5]}")

Total number of words: 32033
Shortest word length : 2
Longest word length  : 15
First 5 words        : ['emma', 'olivia', 'ava', 'isabella', 'sophia']


### Character Mapping & Indexing

What this cell does: Creates bi-directional lookup maps to convert characters to integers (stoi) and integers back to characters (itos). <br> Key Inner Mechanics to Note: 
- We use a single special character . (assigned index 0) to denote both the Start of Word (<START>) and End of Word (<END>).
- Letters a through z are assigned indices 1 through 26 (total vocabulary size $N = 27$).

In [2]:
# Extract unique characters and sort them
chars = sorted(list(set(''.join(words))))

# Create mappings (using '.' as both start and end token at index 0)
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0

itos = {i: s for s, i in stoi.items()}

vocab_size = len(stoi)
print(f"Vocabulary Size: {vocab_size}")
print(f"Mapping sample : {list(stoi.items())[:5]}")

Vocabulary Size: 27
Mapping sample : [('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5)]


### Approach 1 — Bigram Counting Matrix <br> 
What this cell does: Builds a $27 \times 27$ frequency matrix N using PyTorch tensors. We iterate over every adjacent bigram pair in the dataset (including start and end transitions with .) and increment the co-occurrence count at position N[row, col].

In [3]:
# Create 27x27 count matrix initialized to zeros
N = torch.zeros((27, 27), dtype=torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

print(f"Count matrix shape: {N.shape}")
print("Sample row (counts following 'a'):")
print(N[stoi['a']])

Count matrix shape: torch.Size([27, 27])
Sample row (counts following 'a'):
tensor([6640,  556,  541,  470, 1042,  692,  134,  168, 2332, 1650,  175,  568,
        2528, 1634, 5438,   63,   82,   60, 3264, 1118,  687,  381,  834,  161,
         182, 2050,  435], dtype=torch.int32)


### Normalization & Laplace Smoothing <br> 
What this cell does: Normalizes raw bigram counts into a row-wise probability matrix P. <br> 
Key Inner Mechanics to Note: 
- Laplace Smoothing (+ 1): Adding 1 to raw counts prevents zero-probability transitions ($P=0$), which would cause $\log(0) = -\infty$ loss during evaluation.
- Broadcast Safety (keepdim=True): Summing along dimension 1 with keepdim=True creates a column vector of shape (27, 1), allowing PyTorch to correctly divide each 27x27 row by its respective row sum.

In [4]:
# Add 1 count for Laplace smoothing to avoid zero probabilities
P = (N + 1).float()

# Normalize along rows so each row sums to 1.0
P /= P.sum(dim=1, keepdim=True)

print(f"Probability matrix shape: {P.shape}")
print(f"Row 'a' sums to: {P[stoi['a']].sum().item():.4f}")

Probability matrix shape: torch.Size([27, 27])
Row 'a' sums to: 1.0000


### Sampling Names from the Count-Based Model
What this cell does: Generates new synthetic names using torch.multinomial. Starting at character index 0 (.), it samples the next character according to row P[current_index], prints the character, and continues until it samples 0 (.) again.

In [5]:
# Set seed for reproducible sampling
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0  # Start at '.'
    
    while True:
        p = P[ix]
        # Sample next character index from probability distribution p
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:  # Reached end token '.'
            break
            
    print(''.join(out[:-1]))  # Omit trailing '.'

cexze
momasurailezitynn
konimittain
llayn
ka


### Measuring Loss (Negative Log Likelihood) <br> 
What this cell does: Evaluates the quality of our count-based model across the entire dataset using Negative Log-Likelihood (NLL) loss. <br> 
Key Inner Mechanics to Note:
- Likelihood: Product of all predicted probabilities for actual transitions.
- Log-Likelihood: Sum of $\log(P)$. Higher (closer to 0) is better.
- NLL (Loss): Inverted and averaged ($-\frac{1}{N} \sum \log P$). Lower is better.

In [6]:
log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        log_likelihood += torch.log(prob)
        n += 1

nll = -log_likelihood
mean_nll = nll / n

print(f"Total bigrams evaluated : {n}")
print(f"Log Likelihood          : {log_likelihood.item():.4f}")
print(f"Negative Log Likelihood : {nll.item():.4f}")
print(f"Average Loss (NLL)      : {mean_nll.item():.4f}")

Total bigrams evaluated : 228146
Log Likelihood          : -559951.5625
Negative Log Likelihood : 559951.5625
Average Loss (NLL)      : 2.4544


Approach 2 — Preparing Dataset for Neural Network <br>
What this cell does: Formulates bigram modeling as a supervised classification problem. We extract input indices (xs) and corresponding target output indices (ys) for every adjacent character pair.

In [7]:
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(f"Total training pairs: {xs.nelement()}")
print(f"First 5 inputs (xs) : {xs[:5]}")
print(f"First 5 targets (ys): {ys[:5]}")

Total training pairs: 228146
First 5 inputs (xs) : tensor([ 0,  5, 13, 13,  1])
First 5 targets (ys): tensor([ 5, 13, 13,  1,  0])


#### One-Hot Encoding & Linear Layer Setup
What this cell does: Converts integer inputs into One-Hot vectors (F.one_hot) and initializes a weight matrix W of shape (27, 27) with random normal distribution values. <br>
Key Inner Mechanics to Note:
- Multiplying a One-Hot vector $x \in \mathbb{R}^{1 \times 27}$ by Weight matrix $W \in \mathbb{R}^{27 \times 27}$ effectively plucks out row $i$ of $W$.
- We enable requires_grad=True on W to allow gradient updates via PyTorch's autograd.

In [8]:
import torch.nn.functional as F

# Initialize generator and random weight matrix (27 inputs -> 27 outputs)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# Example forward step on single batch:
xenc = F.one_hot(xs[:5], num_classes=27).float()  # One-hot encode inputs
logits = xenc @ W                                  # Linear layer (predict raw scores)

print(f"Input batch shape  : {xenc.shape}")
print(f"Output logits shape: {logits.shape}")

Input batch shape  : torch.Size([5, 27])
Output logits shape: torch.Size([5, 27])


#### Neural Network Training Loop (Softmax + NLL + L2 Regularization)
What this cell does: Runs gradient descent optimization on weight matrix W for 100 epochs.

Key Inner Mechanics to Note:
- Softmax: counts = logits.exp(), probs = counts / counts.sum(1, keepdim=True). Converts raw scores to valid probabilities.
- Loss: Selects predicted probabilities for true labels ys using loss = -probs[torch.arange(num), ys].log().mean().
- L2 Regularization (0.01 * (W**2).mean()): Penalizes large weights, driving predictions toward uniform distributions (equivalent to Laplace smoothing).

In [9]:
# Re-initialize weight matrix
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

num_samples = xs.nelement()
learning_rate = 50.0

for k in range(100):
    # 1. Forward Pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W                             # Compute unnormalized log-counts
    counts = logits.exp()                         # Exponentiate logits to get positive counts
    probs = counts / counts.sum(1, keepdim=True)  # Softmax normalization
    
    # Calculate NLL loss + L2 regularization weight decay
    loss = -probs[torch.arange(num_samples), ys].log().mean() + 0.01 * (W**2).mean()
    
    # 2. Backward Pass
    W.grad = None  # Zero gradients
    loss.backward()
    
    # 3. Parameter Update Step (SGD)
    W.data += -learning_rate * W.grad
    
    if k % 10 == 0 or k == 99:
        print(f"Epoch {k:2d} | Loss: {loss.item():.4f}")

Epoch  0 | Loss: 3.7686
Epoch 10 | Loss: 2.6965
Epoch 20 | Loss: 2.5823
Epoch 30 | Loss: 2.5414
Epoch 40 | Loss: 2.5213
Epoch 50 | Loss: 2.5099
Epoch 60 | Loss: 2.5027
Epoch 70 | Loss: 2.4979
Epoch 80 | Loss: 2.4944
Epoch 90 | Loss: 2.4919
Epoch 99 | Loss: 2.4901


#### Sampling Names from the Neural Network Model
What this cell does: Generates sample names from the trained neural network model. It processes inputs through one-hot encoding, linear transformation @ W, and Softmax normalization to yield identical sampling behavior to the counting model.

In [10]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0  # Start at '.'
    
    while True:
        # Forward pass for single character
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)
        
        # Sample next character
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
            
    print(''.join(out[:-1]))

cexze
momasurailezityha
konimittain
llayn
ka
